In [66]:
import pandas as pd

Reading the dataset 

In [67]:
df = pd.read_csv("Data/tn_nashville_2020_04_01.csv")
df_clean = df.copy()

/var/folders/gh/bgll36gx3nggnnyngrrxd7kr0000gn/T/ipykernel_13983/1481283376.py:1: DtypeWarning: Columns (6,8,15,16,17,22,23,24,25,29,30,31,32,33,35,36,37,38,40,41) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("Data/tn_nashville_2020_04_01.csv")


Doing some data inspections to better understand the variables

In [68]:
df["subject_race"].unique()
df["subject_sex"].unique()
df["search_conducted"].unique()

array([False, True, nan], dtype=object)

In [91]:
len(df["search_conducted"])

3092351

In [69]:
list(df_clean.columns)


['raw_row_number',
 'date',
 'time',
 'location',
 'lat',
 'lng',
 'precinct',
 'reporting_area',
 'zone',
 'subject_age',
 'subject_race',
 'subject_sex',
 'officer_id_hash',
 'type',
 'violation',
 'arrest_made',
 'citation_issued',
 'warning_issued',
 'outcome',
 'contraband_found',
 'contraband_drugs',
 'contraband_weapons',
 'frisk_performed',
 'search_conducted',
 'search_person',
 'search_vehicle',
 'search_basis',
 'reason_for_stop',
 'vehicle_registration_state',
 'notes',
 'raw_verbal_warning_issued',
 'raw_written_warning_issued',
 'raw_traffic_citation_issued',
 'raw_misd_state_citation_issued',
 'raw_suspect_ethnicity',
 'raw_driver_searched',
 'raw_passenger_searched',
 'raw_search_consent',
 'raw_search_arrest',
 'raw_search_warrant',
 'raw_search_inventory',
 'raw_search_plain_view']

In [70]:
df_clean.dtypes

raw_row_number                     object
date                               object
time                               object
location                           object
lat                               float64
lng                               float64
precinct                           object
reporting_area                    float64
zone                               object
subject_age                       float64
subject_race                       object
subject_sex                        object
officer_id_hash                    object
type                               object
violation                          object
arrest_made                        object
citation_issued                    object
warning_issued                     object
outcome                            object
contraband_found                   object
contraband_drugs                   object
contraband_weapons                 object
frisk_performed                    object
search_conducted                  

Here I'm cleaning the dataset up by dropping missing outcome variables

In [71]:
df_clean = df_clean.dropna(subset=["search_conducted"])
df_clean["search_conducted"] = df_clean["search_conducted"].astype(int)

Remove implausible ages to reduce noise and ensure the model is trained on realistic demographic values relevant to traffic stop decision-making

In [72]:
df_clean = df_clean[
    (df_clean["subject_age"] >= 16) &
    (df_clean["subject_age"] <= 100)
]

Setting up Model 1: Logistic Regression

In [73]:
from sklearn.model_selection import train_test_split

In [74]:
y = df_clean["search_conducted"]
features = [
    "subject_age",
    "subject_sex",
    "subject_race",
    "date",
    "time",
    "reason_for_stop",
    "violation",
    "precinct",
    "vehicle_registration_state",
]

X = df_clean[features]

Now to do the training /testing split

In [75]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    stratify=y,
    random_state=42
)

Pre processing and logistic regression pipeline

In [76]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


In [77]:
from sklearn.impute import SimpleImputer

# New categorical transformer that handles missing values automatically
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", categorical_transformer, categorical_features) # Use the new transformer here
    ]
)

In [78]:
numeric_features = [
    "subject_age"
]

categorical_features = [
    "subject_sex",
    "subject_race",
    "reason_for_stop",
    "time",
    "date",
    "violation",
    "precinct",
    "vehicle_registration_state"
]

In [79]:
# Ensure categorical columns are strings and handle missing
for col in categorical_features:
    X_train[col] = X_train[col].fillna("missing").astype(str)
    X_test[col] = X_test[col].fillna("missing").astype(str)

In [80]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

Build a pipeline that ensures identical preprocessing is applied during training and evaluation, preventing data leakage

In [81]:
log_reg_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            solver="liblinear",  # stable for smaller datasets
            random_state=42
        ))
    ]
)

Now to fit the model and evaluate

In [82]:
from sklearn.metrics import roc_auc_score, classification_report

log_reg_pipeline.fit(X_train, y_train)

y_pred = log_reg_pipeline.predict(X_test)
y_pred_proba = log_reg_pipeline.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC–AUC: {roc_auc:.3f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

ROC–AUC: 0.749

Classification Report:
              precision    recall  f1-score   support

           0       0.96      1.00      0.98    888907
           1       0.33      0.00      0.00     38216

    accuracy                           0.96    927123
   macro avg       0.64      0.50      0.49    927123
weighted avg       0.93      0.96      0.94    927123



Model 2: Random Forests

In [83]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

# 1. Define the Preprocessing for Random Forest
# We use OrdinalEncoder for categoricals to keep the feature space manageable
rf_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy='median'), numeric_features),
        ("cat", Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
        ]), categorical_features)
    ]
)

# 2. Build the Random Forest Pipeline
rf_pipeline = Pipeline(
    steps=[
        ("preprocessing", rf_preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            max_depth=10,       # Limited depth to prevent overfitting on date/time
            random_state=42,
            n_jobs=-1           # Uses all CPU cores for faster training
        ))
    ]
)

# 3. Fit the model
# NOTE: Ensure you ran the train_test_split cell just before this
rf_pipeline.fit(X_train, y_train)

# 4. Evaluate
y_pred_rf = rf_pipeline.predict(X_test)
y_pred_proba_rf = rf_pipeline.predict_proba(X_test)[:, 1]

print(f"Random Forest ROC–AUC: {roc_auc_score(y_test, y_pred_proba_rf):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

Random Forest ROC–AUC: 0.760

Classification Report:
              precision    recall  f1-score   support

           0       0.96      1.00      0.98    888907
           1       0.67      0.00      0.00     38216

    accuracy                           0.96    927123
   macro avg       0.81      0.50      0.49    927123
weighted avg       0.95      0.96      0.94    927123



Now for 3rd and final model: XG boost

In [84]:
from xgboost import XGBClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

# 1. Define Preprocessing
# Tree models like XGBoost don't need scaling, but they need numbers instead of strings
xgb_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy='median'), numeric_features),
        ("cat", Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
        ]), categorical_features)
    ]
)

# 2. Build the XGBoost Pipeline
xgb_pipeline = Pipeline(
    steps=[
        ("preprocessing", xgb_preprocessor),
        ("classifier", XGBClassifier(
            n_estimators=100,
            max_depth=6,             # XGBoost usually performs better with shallower trees than RF
            learning_rate=0.1,       # The "shrinkage" factor
            random_state=42,
            use_label_encoder=False, 
            eval_metric='logloss',    # Silences a common warning
            n_jobs=-1
        ))
    ]
)

# 3. Fit the model
xgb_pipeline.fit(X_train, y_train)

# 4. Evaluate
y_pred_xgb = xgb_pipeline.predict(X_test)
y_pred_proba_xgb = xgb_pipeline.predict_proba(X_test)[:, 1]

print(f"XGBoost ROC–AUC: {roc_auc_score(y_test, y_pred_proba_xgb):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

/opt/anaconda3/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [17:04:38] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost ROC–AUC: 0.773

Classification Report:
              precision    recall  f1-score   support

           0       0.96      1.00      0.98    888907
           1       0.54      0.00      0.00     38216

    accuracy                           0.96    927123
   macro avg       0.75      0.50      0.49    927123
weighted avg       0.94      0.96      0.94    927123

